In [ ]:
import re
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# =========================
# 1. 파일 읽기
# =========================
FILE_PATH = "민원_테스트데이터_1250row.xlsx"   # 네 파일명에 맞게 수정 가능

df = pd.read_excel(FILE_PATH, sheet_name="dataset")

print("원본 shape:", df.shape)
print(df.head(3))


# =========================
# 2. 필요한 컬럼만 사용
# =========================
use_cols = ["민원 요청문장", "민원명", "담당기관(청)", "담당부서(과)", "필요문서"]
df = df[use_cols].copy()

# 내부 처리용으로 컬럼명 변경
df.columns = ["text", "complaint", "org", "dept", "docs"]


# =========================
# 3. 기본 결측치 제거
# =========================
# text, complaint는 반드시 있어야 함
df = df.dropna(subset=["text", "complaint"]).copy()

# org, dept, docs가 비어 있으면 빈 문자열로 처리
for col in ["org", "dept", "docs"]:
    df[col] = df[col].fillna("")


# =========================
# 4. 텍스트 정리 함수
# =========================
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 줄바꿈, 탭 제거
    text = text.replace("\n", " ").replace("\t", " ")

    # 앞뒤 공백 제거
    text = text.strip()

    # 연속 공백 -> 한 칸
    text = re.sub(r"\s+", " ", text)

    # 과도한 반복 특수문자 정리
    text = re.sub(r"[!]{2,}", "!", text)
    text = re.sub(r"[?]{2,}", "?", text)
    text = re.sub(r"[.]{2,}", ".", text)

    # ㅋㅋㅋㅋ / ㅎㅎㅎㅎ / ㅠㅠㅠㅠ / ㅜㅜㅜㅜ 같은 반복 축소
    text = re.sub(r"(ㅋ)\1{2,}", r"ㅋㅋ", text)
    text = re.sub(r"(ㅎ)\1{2,}", r"ㅎㅎ", text)
    text = re.sub(r"(ㅠ)\1{2,}", r"ㅠㅠ", text)
    text = re.sub(r"(ㅜ)\1{2,}", r"ㅜㅜ", text)

    return text


# =========================
# 5. 라벨 문자열 표준화 함수
# =========================
def normalize_label(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x


# =========================
# 6. 필요문서 파싱
# 예: "신분증; 전입신고서; 임대차계약서(해당 시)"
# -> ["신분증", "전입신고서", "임대차계약서(해당 시)"]
# =========================
def split_docs(doc_str: str):
    if not isinstance(doc_str, str) or doc_str.strip() == "":
        return []

    parts = doc_str.split(";")
    parts = [normalize_label(p) for p in parts]
    parts = [p for p in parts if p != ""]
    return parts


# =========================
# 7. 전처리 적용
# =========================
df["text"] = df["text"].apply(clean_text)
df["complaint"] = df["complaint"].apply(normalize_label)
df["org"] = df["org"].apply(normalize_label)
df["dept"] = df["dept"].apply(normalize_label)
df["doc_list"] = df["docs"].apply(split_docs)

# 빈 텍스트 제거
df = df[df["text"] != ""].copy()


# =========================
# 8. 완전 중복 제거
# text + complaint + org + dept + docs 기준
# =========================
before = len(df)
df = df.drop_duplicates(subset=["text", "complaint", "org", "dept", "docs"]).copy()
after = len(df)

print(f"중복 제거: {before} -> {after}")


# =========================
# 9. 멀티레이블용 labels 리스트 생성
# prefix를 붙여서 라벨 종류를 구분
# =========================
def make_labels(row):
    labels = []

    if row["complaint"]:
        labels.append(f"민원명:{row['complaint']}")
    if row["org"]:
        labels.append(f"기관:{row['org']}")
    if row["dept"]:
        labels.append(f"부서:{row['dept']}")

    for d in row["doc_list"]:
        labels.append(f"문서:{d}")

    # 중복 제거 + 정렬(재현성)
    labels = sorted(list(set(labels)))
    return labels

df["labels"] = df.apply(make_labels, axis=1)

print(df[["text", "labels"]].head(3))


# =========================
# 10. train / valid / test 분리
# 멀티레이블 stratify는 복잡해서
# 여기서는 complaint(민원명) 기준으로 분리
# =========================
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["complaint"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["complaint"]
)

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)


# =========================
# 11. MultiLabelBinarizer 적용
# 반드시 train으로 fit
# =========================
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["labels"])
y_valid = mlb.transform(valid_df["labels"])
y_test = mlb.transform(test_df["labels"])

print("전체 라벨 수:", len(mlb.classes_))
print("예시 라벨들:", mlb.classes_[:10])


# =========================
# 12. 저장용 데이터프레임 만들기
# text + labels + label vector
# =========================
label_cols = list(mlb.classes_)

train_label_df = pd.DataFrame(y_train, columns=label_cols, index=train_df.index)
valid_label_df = pd.DataFrame(y_valid, columns=label_cols, index=valid_df.index)
test_label_df = pd.DataFrame(y_test, columns=label_cols, index=test_df.index)

train_out = pd.concat(
    [train_df[["text", "complaint", "org", "dept", "docs", "labels"]], train_label_df],
    axis=1
).reset_index(drop=True)

valid_out = pd.concat(
    [valid_df[["text", "complaint", "org", "dept", "docs", "labels"]], valid_label_df],
    axis=1
).reset_index(drop=True)

test_out = pd.concat(
    [test_df[["text", "complaint", "org", "dept", "docs", "labels"]], test_label_df],
    axis=1
).reset_index(drop=True)


# =========================
# 13. 파일 저장
# =========================
train_out.to_csv("train_multilabel.csv", index=False, encoding="utf-8-sig")
valid_out.to_csv("valid_multilabel.csv", index=False, encoding="utf-8-sig")
test_out.to_csv("test_multilabel.csv", index=False, encoding="utf-8-sig")

# 라벨 클래스 저장
with open("multilabel_classes.json", "w", encoding="utf-8") as f:
    json.dump(label_cols, f, ensure_ascii=False, indent=2)

# labels만 간단히 저장한 버전도 같이 저장
train_df[["text", "labels"]].to_json("train_labels_only.json", force_ascii=False, orient="records", indent=2)
valid_df[["text", "labels"]].to_json("valid_labels_only.json", force_ascii=False, orient="records", indent=2)
test_df[["text", "labels"]].to_json("test_labels_only.json", force_ascii=False, orient="records", indent=2)

print("저장 완료!")
print("- train_multilabel.csv")
print("- valid_multilabel.csv")
print("- test_multilabel.csv")
print("- multilabel_classes.json")
print("- train_labels_only.json / valid_labels_only.json / test_labels_only.json")